# ╔══════════════════════════════════════════════════════════════╗
# ║   PERFECT 4-MODEL ENSEMBLE FUSION — FINAL PREDICTIONS       ║
# ║   Models: rPPG | EfficientNet-B4 | Xception | Swin-Tiny     ║
# ╚══════════════════════════════════════════════════════════════╝
#
# EXACT FILE NAMES FROM EACH NOTEBOOK:
#
# 1. model_rppg.ipynb        → rppg_predictions.csv          (col: P_rPPG)
# 2. model_efficientnet.ipynb→ cnn_predictions.csv           (col: P_CNN)
# 3. model_xception-1.ipynb  → cnn_predictions.csv           (col: P_CNN)
# 4. model_swin-1.ipynb      → cnn_predictions_swin_oof_MASTER.csv  (col: P_CNN)
#
# HOW TO USE ON KAGGLE:
# - Run each model notebook first and download its /kaggle/working/ output as a dataset
# - Add each downloaded dataset as a Kaggle input to THIS notebook
# - Update the INPUT PATHS section below to match your dataset names
# ─────────────────────────────────────────────────────────────────

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 1: IMPORTS
# ═══════════════════════════════════════════════════════════════════════════════

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.metrics import (
    accuracy_score, roc_auc_score, f1_score,
    precision_score, recall_score, roc_curve,
    confusion_matrix, classification_report,
    average_precision_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from scipy.stats import rankdata

SEED = 42
np.random.seed(SEED)

OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('✓ All imports loaded')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 2: INPUT PATHS — UPDATE THESE TO MATCH YOUR KAGGLE DATASET NAMES
# ═══════════════════════════════════════════════════════════════════════════════
#
# Each model's /kaggle/working/ must be uploaded as a separate Kaggle dataset.
# Then add each dataset as input to this notebook and update paths below.
#
# Example:
#   If you uploaded rPPG outputs as dataset 'my-rppg-outputs',
#   RPPG_CSV = '/kaggle/input/my-rppg-outputs/rppg_predictions.csv'
# ─────────────────────────────────────────────────────────────────────────────

# ── model_rppg.ipynb output ──────────────────────────────────────────────────
# Exact filename written by Cell 27 of rPPG notebook: rppg_predictions.csv
# Column used: 'P_rPPG'
RPPG_CSV = '/kaggle/input/rppg-model-outputs/rppg_predictions.csv'

# ── model_efficientnet.ipynb output ──────────────────────────────────────────
# Exact filename written by Cell 33 of EfficientNet notebook: cnn_predictions.csv
# Column used: 'P_CNN'  → will be renamed to 'P_efficientnet'
EFFICIENTNET_CSV = '/kaggle/input/efficientnet-model-outputs/cnn_predictions.csv'

# ── model_xception-1.ipynb output ────────────────────────────────────────────
# Exact filename written by Cell 35 of Xception notebook: cnn_predictions.csv
# Column used: 'P_CNN'  → will be renamed to 'P_xception'
XCEPTION_CSV = '/kaggle/input/xception-model-outputs/cnn_predictions.csv'

# ── model_swin-1.ipynb output ────────────────────────────────────────────────
# Exact filename written by Cell 24 of Swin notebook: cnn_predictions_swin_oof_MASTER.csv
# This is the merged OOF master across ALL 5 folds.
# Column used: 'P_CNN'  → will be renamed to 'P_swin'
# ALTERNATIVE (single fold): 'cnn_predictions_swin_fold0.csv'
SWIN_CSV = '/kaggle/input/swin-model-outputs/cnn_predictions_swin_oof_MASTER.csv'

# ── Print configuration ───────────────────────────────────────────────────────
print('═'*70)
print('INPUT FILE CONFIGURATION')
print('═'*70)
for name, path in [
    ('rPPG',         RPPG_CSV),
    ('EfficientNet', EFFICIENTNET_CSV),
    ('Xception',     XCEPTION_CSV),
    ('Swin-Tiny',    SWIN_CSV)
]:
    exists = '✓' if os.path.exists(path) else '✗ NOT FOUND'
    print(f'  {exists}  {name:15s}: {path}')
print('═'*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 3: LOAD & VALIDATE ALL PREDICTION FILES
# ═══════════════════════════════════════════════════════════════════════════════

def load_predictions(csv_path, score_col, model_name, rename_to):
    """
    Load a predictions CSV, validate required columns, and rename score column.
    
    Every prediction CSV from all 4 notebooks contains:
        - 'video_id'  : unique string identifier matching master_dataset_index.csv
        - 'label'     : ground truth (0=Real, 1=Fake)
        - score_col   : float probability of being Fake
    """
    if not os.path.exists(csv_path):
        raise FileNotFoundError(
            f"❌ {model_name} predictions not found at: {csv_path}\n"
            f"   Please run {model_name} notebook first and upload its output."
        )
    
    df = pd.read_csv(csv_path)
    
    # Validate required columns
    for col in ['video_id', score_col]:
        if col not in df.columns:
            raise ValueError(
                f"❌ {model_name}: missing column '{col}'. "
                f"Found: {list(df.columns)}"
            )
    
    # 'label' may not be present in all files — tolerate its absence
    has_label = 'label' in df.columns
    
    # Rename score column to avoid collision when merging
    df = df.rename(columns={score_col: rename_to})
    
    # Clean up: remove duplicates (Swin OOF can have dupes from fold overlap)
    before = len(df)
    df = df.drop_duplicates(subset='video_id', keep='last')
    if len(df) < before:
        print(f'  ⚠ {model_name}: removed {before - len(df)} duplicate video_ids')
    
    # Clamp probabilities to [0, 1]
    df[rename_to] = df[rename_to].clip(0.0, 1.0)
    
    # Report
    label_info = ''
    if has_label:
        label_info = f' | Real={int((df["label"]==0).sum())} Fake={int((df["label"]==1).sum())}'
    print(f'  ✓ {model_name:15s}: {len(df):5d} videos{label_info}')
    
    return df


print('Loading predictions from all 4 models...')
print('─'*60)

df_rppg  = load_predictions(RPPG_CSV,         'P_rPPG', 'rPPG',         'P_rPPG')
df_eff   = load_predictions(EFFICIENTNET_CSV,  'P_CNN',  'EfficientNet', 'P_efficientnet')
df_xcep  = load_predictions(XCEPTION_CSV,      'P_CNN',  'Xception',     'P_xception')
df_swin  = load_predictions(SWIN_CSV,          'P_CNN',  'Swin-Tiny',    'P_swin')

print('─'*60)
print('✓ All prediction files loaded successfully')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 4: ALIGN & MERGE ON video_id
# ═══════════════════════════════════════════════════════════════════════════════
#
# All 4 notebooks use video_id from master_dataset_index.csv.
# Format: '{source}__{relative_path}' e.g. 'FF_real__001_870.mp4'
# rPPG uses basename of video paths as video_id via paths_array.
#
# We do an INNER JOIN so only videos present in ALL 4 models are kept.
# This guarantees a fair comparison.
# ─────────────────────────────────────────────────────────────────────────────

# Keep only relevant columns from each dataframe before merging
rppg_cols  = ['video_id', 'P_rPPG']  + (['label'] if 'label' in df_rppg.columns  else [])
eff_cols   = ['video_id', 'P_efficientnet'] + (['label'] if 'label' in df_eff.columns  else [])
xcep_cols  = ['video_id', 'P_xception']     + (['label'] if 'label' in df_xcep.columns else [])
swin_cols  = ['video_id', 'P_swin']         + (['label'] if 'label' in df_swin.columns else [])

# Determine which df has the ground-truth label column (use first found)
# All notebooks save label from the same master_dataset_index.csv — they match.
label_source = None
for df_check, name in [(df_rppg, 'rPPG'), (df_eff, 'EfficientNet'),
                        (df_xcep, 'Xception'), (df_swin, 'Swin-Tiny')]:
    if 'label' in df_check.columns:
        label_source = name
        break

if label_source is None:
    raise ValueError(
        '❌ No prediction CSV contains a "label" column.\n'
        '   At least one CSV must have the ground-truth labels for evaluation.'
    )

# Use rPPG label as canonical (it covers the widest video set)
# If label conflicts exist (impossible if all from same master CSV), inner join resolves it.

# Step-by-step merge to track attrition
print('Merging predictions on video_id (inner join)...')
print(f'  rPPG         : {len(df_rppg):5d} videos')

merged = df_rppg[rppg_cols].copy()

merged = merged.merge(df_eff[eff_cols].rename(columns={'label': 'label_eff'}),
                      on='video_id', how='inner')
print(f'  After ∩ EfficientNet: {len(merged):5d} videos')

merged = merged.merge(df_xcep[xcep_cols].rename(columns={'label': 'label_xcep'}),
                      on='video_id', how='inner')
print(f'  After ∩ Xception    : {len(merged):5d} videos')

merged = merged.merge(df_swin[swin_cols].rename(columns={'label': 'label_swin'}),
                      on='video_id', how='inner')
print(f'  After ∩ Swin-Tiny   : {len(merged):5d} videos')

# Reconcile labels: all should agree (they all came from the same master CSV)
label_cols = [c for c in merged.columns if c.startswith('label')]
if len(label_cols) > 1:
    # Check consistency
    label_matrix = merged[label_cols].values
    disagreements = (label_matrix != label_matrix[:, :1]).any(axis=1).sum()
    if disagreements > 0:
        print(f'  ⚠ WARNING: {disagreements} label disagreements across models!')
        print('    This should not happen if all models used the same master_dataset_index.csv')
    # Use the first available label column
    merged['label'] = merged[label_cols[0]]
    merged.drop(columns=[c for c in label_cols if c != 'label'], inplace=True)

# Final check
assert merged['label'].notna().all(), '❌ NaN values in label column'
merged['label'] = merged['label'].astype(int)

SCORE_COLS = ['P_rPPG', 'P_efficientnet', 'P_xception', 'P_swin']

# NaN check on scores
nan_counts = merged[SCORE_COLS].isna().sum()
if nan_counts.any():
    print(f'\n  ⚠ NaN scores found: {nan_counts[nan_counts>0].to_dict()}')
    print('    Filling with 0.5 (neutral prediction)')
    merged[SCORE_COLS] = merged[SCORE_COLS].fillna(0.5)

print('\n' + '='*60)
print('FINAL ALIGNED DATASET')
print('='*60)
print(f'  Total samples : {len(merged)}')
print(f'  Real (label=0): {int((merged["label"]==0).sum())}')
print(f'  Fake (label=1): {int((merged["label"]==1).sum())}')
print(f'  Score columns : {SCORE_COLS}')
print('='*60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 5: INDIVIDUAL MODEL METRICS
# ═══════════════════════════════════════════════════════════════════════════════

y_true = merged['label'].values
MODEL_LABELS = {
    'P_rPPG':         'rPPG (ML Stacking)',
    'P_efficientnet': 'EfficientNet-B4 (BiLSTM+Attn)',
    'P_xception':     'Xception (BiLSTM+Freq)',
    'P_swin':         'Swin-Tiny (BiLSTM+DCT)',
}

individual_aucs = {}
print('═'*70)
print('INDIVIDUAL MODEL PERFORMANCE')
print('═'*70)
print(f"{'Model':35s} {'AUC':>7} {'Acc':>7} {'F1':>7} {'Prec':>7} {'Rec':>7}")
print('─'*70)

for col in SCORE_COLS:
    probs = merged[col].values
    preds = (probs >= 0.5).astype(int)
    auc   = roc_auc_score(y_true, probs)
    acc   = accuracy_score(y_true, preds)
    f1    = f1_score(y_true, preds, zero_division=0)
    prec  = precision_score(y_true, preds, zero_division=0)
    rec   = recall_score(y_true, preds, zero_division=0)
    individual_aucs[col] = auc
    print(f"  {MODEL_LABELS[col]:33s} {auc:7.4f} {acc:7.4f} {f1:7.4f} {prec:7.4f} {rec:7.4f}")

print('═'*70)
print(f'  Best individual AUC: {max(individual_aucs.values()):.4f} '
      f'({MODEL_LABELS[max(individual_aucs, key=individual_aucs.get)]})')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 6: ENSEMBLE METHODS
# ═══════════════════════════════════════════════════════════════════════════════
#
# Method 1: Simple Average
# Method 2: AUC-Weighted Average (weights = individual AUC / sum(AUCs))
# Method 3: Rank-Based Ensemble (robust to scale differences)
# Method 4: Learned Meta-Learner (LogisticRegression on out-of-fold probs)
# ─────────────────────────────────────────────────────────────────────────────

X_scores = merged[SCORE_COLS].values  # shape: (N, 4)
ensemble_results = {}

def eval_ensemble(probs, name, y_true):
    preds = (probs >= 0.5).astype(int)
    auc   = roc_auc_score(y_true, probs)
    acc   = accuracy_score(y_true, preds)
    f1    = f1_score(y_true, preds, zero_division=0)
    prec  = precision_score(y_true, preds, zero_division=0)
    rec   = recall_score(y_true, preds, zero_division=0)
    ensemble_results[name] = dict(auc=auc, acc=acc, f1=f1, prec=prec, rec=rec, probs=probs)
    return auc, acc, f1


print('═'*70)
print('ENSEMBLE METHODS')
print('═'*70)
print(f"{'Method':35s} {'AUC':>7} {'Acc':>7} {'F1':>7}")
print('─'*70)


# ── Method 1: Simple Average ──────────────────────────────────────────────────
P_simple = X_scores.mean(axis=1)
auc, acc, f1 = eval_ensemble(P_simple, 'simple_avg', y_true)
print(f"  {'Simple Average':33s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")


# ── Method 2: AUC-Weighted Average ───────────────────────────────────────────
auc_weights = np.array([individual_aucs[c] for c in SCORE_COLS])
auc_weights = auc_weights / auc_weights.sum()
P_weighted = (X_scores * auc_weights[None, :]).sum(axis=1)
auc, acc, f1 = eval_ensemble(P_weighted, 'auc_weighted', y_true)
print(f"  {'AUC-Weighted Average':33s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
print(f"    Weights: rPPG={auc_weights[0]:.3f} EfficientNet={auc_weights[1]:.3f} "
      f"Xception={auc_weights[2]:.3f} Swin={auc_weights[3]:.3f}")


# ── Method 3: Rank-Based Ensemble ────────────────────────────────────────────
# Convert each model's probabilities to normalized ranks [0,1]
N = len(y_true)
ranked = np.column_stack([
    rankdata(X_scores[:, i]) / N for i in range(X_scores.shape[1])
])
P_rank = ranked.mean(axis=1)
auc, acc, f1 = eval_ensemble(P_rank, 'rank_based', y_true)
print(f"  {'Rank-Based Average':33s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")


# ── Method 4: Learned Meta-Learner (5-fold CV to avoid leakage) ──────────────
# LogisticRegression trained on the 4 model scores as features.
# Uses 5-fold stratified cross-validation for leakage-free OOF predictions.
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_scores)

meta_lr = LogisticRegression(
    C=1.0, max_iter=1000, random_state=SEED,
    class_weight='balanced'
)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
P_meta_oof = cross_val_predict(
    meta_lr, X_scaled, y_true,
    cv=skf, method='predict_proba'
)[:, 1]
auc, acc, f1 = eval_ensemble(P_meta_oof, 'meta_learner', y_true)
print(f"  {'Meta-Learner (LR, 5-fold OOF)':33s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")

# Fit on full data for final inference
meta_lr.fit(X_scaled, y_true)
print(f"    LR coefficients: rPPG={meta_lr.coef_[0][0]:.3f} "
      f"EfficientNet={meta_lr.coef_[0][1]:.3f} "
      f"Xception={meta_lr.coef_[0][2]:.3f} "
      f"Swin={meta_lr.coef_[0][3]:.3f}")


# ── Method 5: Optimal Grid-Search Weighted Average ───────────────────────────
# Search over weights w1..w4 (sum=1) using 5-fold CV AUC as objective
print(f"\n  Grid-searching optimal weights (this takes ~30 seconds)...")

best_oof_auc = 0.0
best_w = auc_weights.copy()
# Coarse search over 4-model simplex
step = 0.1
w_vals = np.arange(0.0, 1.01, step)

for w0 in w_vals:
    for w1 in w_vals:
        for w2 in w_vals:
            w3 = 1.0 - w0 - w1 - w2
            if w3 < 0 or w3 > 1.0: continue
            w = np.array([w0, w1, w2, w3])
            if w.sum() < 0.999 or w.sum() > 1.001: continue
            P_cand = (X_scores * w[None, :]).sum(axis=1)
            try:
                cand_auc = roc_auc_score(y_true, P_cand)
            except:
                continue
            if cand_auc > best_oof_auc:
                best_oof_auc = cand_auc
                best_w = w.copy()

P_optimal = (X_scores * best_w[None, :]).sum(axis=1)
auc, acc, f1 = eval_ensemble(P_optimal, 'optimal_weighted', y_true)
print(f"  {'Optimal Grid-Search Weights':33s} {auc:7.4f} {acc:7.4f} {f1:7.4f}")
print(f"    Best weights: rPPG={best_w[0]:.2f} EfficientNet={best_w[1]:.2f} "
      f"Xception={best_w[2]:.2f} Swin={best_w[3]:.2f}")


print('═'*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 7: SELECT BEST ENSEMBLE & FIND OPTIMAL THRESHOLD
# ═══════════════════════════════════════════════════════════════════════════════

# Select best method by AUC
best_name = max(ensemble_results, key=lambda k: ensemble_results[k]['auc'])
best_probs = ensemble_results[best_name]['probs']

print('═'*70)
print(f'BEST ENSEMBLE: {best_name.upper().replace("_", " ")}')
print('═'*70)

# Find threshold maximizing F1 (on the full set — same set used for all models)
best_thresh, best_f1 = 0.5, 0.0
for thresh in np.arange(0.05, 0.96, 0.01):
    preds = (best_probs >= thresh).astype(int)
    f1_val = f1_score(y_true, preds, zero_division=0)
    if f1_val > best_f1:
        best_f1 = f1_val
        best_thresh = thresh

final_preds = (best_probs >= best_thresh).astype(int)

final_auc  = roc_auc_score(y_true, best_probs)
final_acc  = accuracy_score(y_true, final_preds)
final_f1   = f1_score(y_true, final_preds, zero_division=0)
final_prec = precision_score(y_true, final_preds, zero_division=0)
final_rec  = recall_score(y_true, final_preds, zero_division=0)
final_ap   = average_precision_score(y_true, best_probs)

print(f'  Method        : {best_name}')
print(f'  Threshold     : {best_thresh:.2f} (F1-optimal)')
print(f'  AUC-ROC       : {final_auc:.4f}')
print(f'  Avg Precision : {final_ap:.4f}')
print(f'  Accuracy      : {final_acc:.4f}')
print(f'  F1-Score      : {final_f1:.4f}')
print(f'  Precision     : {final_prec:.4f}')
print(f'  Recall        : {final_rec:.4f}')
print('═'*70)
print()
print(classification_report(y_true, final_preds,
                            target_names=['Real', 'Fake'], digits=4))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 8: BOOTSTRAP CONFIDENCE INTERVALS (Research-Grade)
# ═══════════════════════════════════════════════════════════════════════════════

def bootstrap_ci(y_true, y_pred, metric_fn, n_boot=1000, ci=0.95, seed=42):
    rng = np.random.RandomState(seed)
    scores = []
    for _ in range(n_boot):
        idx = rng.choice(len(y_true), len(y_true), replace=True)
        yt, yp = y_true[idx], y_pred[idx]
        if len(np.unique(yt)) < 2:
            continue
        try:
            scores.append(metric_fn(yt, yp))
        except Exception:
            continue
    if not scores:
        return 0.0, 0.0, 0.0
    alpha = (1 - ci) / 2
    return (float(np.mean(scores)),
            float(np.percentile(scores, alpha * 100)),
            float(np.percentile(scores, (1 - alpha) * 100)))


print('Computing 95% Bootstrap CIs (n=1000 iterations)...')

auc_m, auc_lo, auc_hi = bootstrap_ci(y_true, best_probs,   roc_auc_score)
acc_m, acc_lo, acc_hi = bootstrap_ci(y_true, final_preds,  accuracy_score)
f1_m,  f1_lo,  f1_hi  = bootstrap_ci(y_true, final_preds,
                                       lambda t, p: f1_score(t, p, zero_division=0))
pre_m, pre_lo, pre_hi = bootstrap_ci(y_true, final_preds,
                                      lambda t, p: precision_score(t, p, zero_division=0))
rec_m, rec_lo, rec_hi = bootstrap_ci(y_true, final_preds,
                                      lambda t, p: recall_score(t, p, zero_division=0))

print('═'*65)
print('FINAL ENSEMBLE — BOOTSTRAP 95% CONFIDENCE INTERVALS')
print('═'*65)
print(f'  AUC-ROC:   {auc_m:.4f}  [{auc_lo:.4f},  {auc_hi:.4f}]')
print(f'  Accuracy:  {acc_m:.4f}  [{acc_lo:.4f},  {acc_hi:.4f}]')
print(f'  F1-Score:  {f1_m:.4f}  [{f1_lo:.4f},   {f1_hi:.4f}]')
print(f'  Precision: {pre_m:.4f}  [{pre_lo:.4f},  {pre_hi:.4f}]')
print(f'  Recall:    {rec_m:.4f}  [{rec_lo:.4f},  {rec_hi:.4f}]')
print('═'*65)

metrics_df = pd.DataFrame([
    {'metric': 'AUC-ROC',   'value': auc_m,  'ci_low': auc_lo,  'ci_high': auc_hi},
    {'metric': 'Accuracy',  'value': acc_m,  'ci_low': acc_lo,  'ci_high': acc_hi},
    {'metric': 'F1-Score',  'value': f1_m,   'ci_low': f1_lo,   'ci_high': f1_hi},
    {'metric': 'Precision', 'value': pre_m,  'ci_low': pre_lo,  'ci_high': pre_hi},
    {'metric': 'Recall',    'value': rec_m,  'ci_low': rec_lo,  'ci_high': rec_hi},
])
metrics_df.to_csv(os.path.join(OUTPUT_DIR, 'ensemble_metrics_with_ci.csv'), index=False)
print('\n✓ Metrics saved → ensemble_metrics_with_ci.csv')

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 9: RESEARCH-GRADE VISUALIZATIONS
# ═══════════════════════════════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# ── Plot 1: ROC Curves — All Models + Ensemble ────────────────────────────────
ax = axes[0, 0]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

for i, col in enumerate(SCORE_COLS):
    fpr, tpr, _ = roc_curve(y_true, merged[col].values)
    auc_val = individual_aucs[col]
    ax.plot(fpr, tpr, color=colors[i], lw=1.5, alpha=0.7,
            label=f'{MODEL_LABELS[col]} ({auc_val:.3f})')

# Best ensemble
fpr_ens, tpr_ens, _ = roc_curve(y_true, best_probs)
ax.plot(fpr_ens, tpr_ens, color=colors[4], lw=3, ls='--',
        label=f'Ensemble ({final_auc:.3f})')
ax.plot([0, 1], [0, 1], 'k:', lw=1, label='Random')
ax.set_xlabel('False Positive Rate', fontsize=11)
ax.set_ylabel('True Positive Rate', fontsize=11)
ax.set_title('ROC Curves — All Models', fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.grid(True, alpha=0.3)


# ── Plot 2: AUC Comparison Bar Chart ─────────────────────────────────────────
ax = axes[0, 1]
model_names = [MODEL_LABELS[c].split(' ')[0] for c in SCORE_COLS]
aucs_plot   = [individual_aucs[c] for c in SCORE_COLS]
model_names_full = model_names + ['Ensemble']
aucs_full        = aucs_plot + [final_auc]
bar_colors = colors[:4] + [colors[4]]
bars = ax.bar(model_names_full, aucs_full, color=bar_colors, alpha=0.85, edgecolor='white')
for bar, auc_val in zip(bars, aucs_full):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{auc_val:.4f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim([max(0, min(aucs_full) - 0.05), min(1.0, max(aucs_full) + 0.05)])
ax.set_ylabel('AUC-ROC', fontsize=11)
ax.set_title('AUC-ROC Comparison', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=15)


# ── Plot 3: Confusion Matrix ──────────────────────────────────────────────────
ax = axes[0, 2]
cm = confusion_matrix(y_true, final_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'],
            annot_kws={'size': 16})
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title(f'Confusion Matrix\n(Threshold={best_thresh:.2f})', fontsize=13, fontweight='bold')


# ── Plot 4: Score Distribution ────────────────────────────────────────────────
ax = axes[1, 0]
ax.hist(best_probs[y_true==0], bins=40, alpha=0.6, color='#4CAF50',
        label='Real', density=True)
ax.hist(best_probs[y_true==1], bins=40, alpha=0.6, color='#F44336',
        label='Fake', density=True)
ax.axvline(x=best_thresh, color='black', lw=2, ls='--',
           label=f'Threshold={best_thresh:.2f}')
ax.set_xlabel('Ensemble Probability P(Fake)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Score Distribution (Best Ensemble)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)


# ── Plot 5: Pairwise Correlation of Model Scores ──────────────────────────────
ax = axes[1, 1]
corr = merged[SCORE_COLS].corr()
short_labels = ['rPPG', 'EffNet', 'Xcep', 'Swin']
corr.columns = short_labels
corr.index   = short_labels
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            ax=ax, square=True, vmin=-1, vmax=1,
            annot_kws={'size': 11})
ax.set_title('Model Score Correlations\n(Lower = More Diverse)', fontsize=13, fontweight='bold')


# ── Plot 6: Ensemble Methods AUC Comparison ───────────────────────────────────
ax = axes[1, 2]
method_names = list(ensemble_results.keys())
method_aucs  = [ensemble_results[m]['auc'] for m in method_names]
display_names = {
    'simple_avg':      'Simple Avg',
    'auc_weighted':    'AUC-Weighted',
    'rank_based':      'Rank-Based',
    'meta_learner':    'Meta-LR (OOF)',
    'optimal_weighted':'Grid-Search',
}
bar_c = ['#2196F3' if m != best_name else '#F44336' for m in method_names]
bars2 = ax.bar([display_names.get(m, m) for m in method_names],
               method_aucs, color=bar_c, alpha=0.85, edgecolor='white')
for bar, auc_val in zip(bars2, method_aucs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{auc_val:.4f}', ha='center', va='bottom', fontsize=9)
ax.set_ylim([max(0, min(method_aucs)-0.03), min(1, max(method_aucs)+0.03)])
ax.set_ylabel('AUC-ROC', fontsize=11)
ax.set_title('Ensemble Methods Comparison\n(Red = Best)', fontsize=13, fontweight='bold')
ax.tick_params(axis='x', rotation=20)
ax.grid(axis='y', alpha=0.3)


plt.suptitle('4-Model Ensemble Fusion: rPPG + EfficientNet-B4 + Xception + Swin-Tiny',
             fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'ensemble_evaluation_plots.png'),
            dpi=200, bbox_inches='tight', facecolor='white')
print('✓ Plots saved → ensemble_evaluation_plots.png')
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 10: SAVE FINAL PREDICTIONS
# ═══════════════════════════════════════════════════════════════════════════════

# Save all ensemble scores + individual scores + ground truth
final_df = merged[['video_id', 'label'] + SCORE_COLS].copy()

# Add all ensemble predictions
final_df['P_simple_avg']      = ensemble_results['simple_avg']['probs']
final_df['P_auc_weighted']    = ensemble_results['auc_weighted']['probs']
final_df['P_rank_based']      = ensemble_results['rank_based']['probs']
final_df['P_meta_learner']    = ensemble_results['meta_learner']['probs']
final_df['P_optimal_weighted']= ensemble_results['optimal_weighted']['probs']

# Best ensemble
final_df['P_final']           = best_probs
final_df['pred_final']        = final_preds
final_df['ensemble_method']   = best_name
final_df['threshold_used']    = best_thresh

out_path = os.path.join(OUTPUT_DIR, 'ensemble_final_predictions.csv')
final_df.to_csv(out_path, index=False)

print('═'*70)
print('FINAL OUTPUT SAVED')
print('═'*70)
print(f'  File    : {out_path}')
print(f'  Rows    : {len(final_df)}')
print(f'  Columns : {list(final_df.columns)}')
print()
print('  Key columns:')
print('    P_final       → Best ensemble probability (use this for final predictions)')
print('    pred_final    → Binary prediction (0=Real, 1=Fake) at optimal threshold')
print('    label         → Ground truth')
print('='*70)
print()
print('SUMMARY TABLE')
print('─'*70)
print(f"{'Model/Method':38s} AUC")
print('─'*70)
for col in SCORE_COLS:
    print(f"  Individual: {MODEL_LABELS[col]:26s} {individual_aucs[col]:.4f}")
print('─'*50)
for method in ensemble_results:
    mark = ' ★ BEST' if method == best_name else ''
    print(f"  Ensemble:   {method:26s} {ensemble_results[method]['auc']:.4f}{mark}")
print('─'*70)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CELL 11: ARCHIVE ALL OUTPUTS
# ═══════════════════════════════════════════════════════════════════════════════

import shutil
shutil.make_archive('ensemble_outputs', 'zip', OUTPUT_DIR)
print('✓ All outputs zipped → ensemble_outputs.zip')
print()
print('Files in /kaggle/working/:')
for f in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
    print(f'  {f:45s}  {size_kb:8.1f} KB')